# This might not have been the exact code that was used to create `zarr-chunk-meta.tsv`, but it gives you an idea of how it was roughly achieved. 

In [ ]:
%%time
# Build chunk index with array indices

import numpy as np
import pandas as pd

pos_var = variant_data["variant_position"]
chrom_var = variant_data["variant_chrom"]
chunk_sizes = pos_var.chunks[0]

chunk_index = []
offset = 0

for i, chunk_size in enumerate(chunk_sizes):
    chunk_start = offset
    chunk_end = offset + chunk_size
    
    # Load positions and chromosomes for this chunk
    chunk_pos = pos_var.isel(variants=slice(chunk_start, chunk_end)).values
    chunk_chrom = chrom_var.isel(variants=slice(chunk_start, chunk_end)).values
    
    # Find unique chromosomes in this chunk
    unique_chroms = np.unique(chunk_chrom)
    
    for chrom in unique_chroms:
        chrom_mask = chunk_chrom == chrom
        chrom_positions = chunk_pos[chrom_mask]
        chrom_indices = np.where(chrom_mask)[0]
        
        chunk_index.append({
            'chunk': i,
            'chunk_start': chunk_start + chrom_indices[0],    # absolute array index
            'chunk_end': chunk_start + chrom_indices[-1] + 1, # absolute array index
            'chrom': chrom,
            'min_pos': int(chrom_positions.min()),
            'max_pos': int(chrom_positions.max()),
        })
    
    offset += chunk_size
    
    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(chunk_sizes)} chunks")

chunk_df = pd.DataFrame(chunk_index)
chunk_df.to_csv("zarr-chunk-meta.tsv", sep="\t", index=False)
print(chunk_df.head(15))